In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import pandas as pd
from src.paths import TRASNFORMED_DATA_DIR

df = pd.read_parquet(TRASNFORMED_DATA_DIR / 'tabular_data.parquet')
df

,rides_previous_672_hours,rides_previous_671_hours,rides_previous_670_hours,rides_previous_669_hours,rides_previous_668_hours,rides_previous_667_hours,rides_previous_666_hours,rides_previous_665_hours,rides_previous_664_hours,rides_previous_663_hours,...,rides_previous_7_hours,rides_previous_6_hours,rides_previous_5_hours,rides_previous_4_hours,rides_previous_3_hours,rides_previous_2_hours,rides_previous_1_hours,pickup_hour,pickup_location_id,target_rides_next_hour
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-01-29,1,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-01-30,1,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,2025-01-31,1,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2025-02-01,1,0.0
4,0.0,1.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-02-02,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88289,0.0,3.0,2.0,4.0,3.0,1.0,2.0,1.0,3.0,3.0,...,4.0,2.0,1.0,1.0,2.0,1.0,2.0,2025-12-27,265,4.0
88290,2.0,5.0,2.0,1.0,6.0,0.0,3.0,0.0,3.0,2.0,...,4.0,8.0,0.0,3.0,3.0,1.0,0.0,2025-12-28,265,1.0
88291,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,8.0,0.0,...,8.0,7.0,1.0,6.0,2.0,7.0,2.0,2025-12-29,265,2.0
88292,1.0,1.0,0.0,1.0,2.0,0.0,1.0,3.0,0.0,0.0,...,3.0,3.0,3.0,4.0,0.0,6.0,1.0,2025-12-30,265,4.0


In [3]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df=df,
    cutoff_data=datetime(2025,6,1,0,0,0),
    target_colum_name='target_rides_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')


X_train.shape=(32226, 674)
y_train.shape=(32226,)
X_test.shape=(56068, 674)
y_test.shape=(56068,)


In [4]:
import numpy as np
import pandas as pd

class BaselineModePreviousHour:
    """
    Prediction = actual demand observed in the last hour
    """
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        pass

    def predict(self, X_test: pd.DataFrame) -> np.array:
        """"""
        return X_test['rides_previous_1_hours'].values


In [5]:
model = BaselineModePreviousHour()
predictions = model.predict(X_test)
predictions

array([0., 0., 0., ..., 2., 1., 0.], shape=(56068,), dtype=float32)

In [6]:
from sklearn.metrics import mean_absolute_error


test_mea = mean_absolute_error(y_test, predictions)
print(f'{test_mea:0.4f}')


8.4421


In [7]:
import numpy as np
import pandas as pd

class BaselineModePreviousWeek:
    """
    Prediction = actual demand observed in the t - 7 days
    """
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        pass

    def predict(self, X_test: pd.DataFrame) -> np.array:
        """"""
        return X_test[f'rides_previous_{7*24}_hours'].values


In [8]:
model = BaselineModePreviousWeek()
predictions = model.predict(X_test)

In [9]:
test_mea = mean_absolute_error(y_test, predictions)
print(f'{test_mea=:0.4f}')

test_mea=6.1555


In [ ]:
class BaselineModelLast4Weeks:
    """
    Prediction = actual demand observes at t - 7 days, t - 14 days, t - 21 days, t - 28 days.
    """
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        pass

    def predict(self, X_test: pd.DataFrame) -> np.array:
        """"""
        return 0.25 * (
            X_test[f'rides_previous_{7*24}_hours'].values +
            X_test[f'rides_previous_{2*7*24}_hours'].values +
            X_test[f'rides_previous_{3*7*24}_hours'].values +
            X_test[f'rides_previous_{4*7*24}_hours'].values 
        )